# Desafio: Analisando Texto sobre Ciência de Dados

Neste exemplo, vamos fazer um exercício simples que cobre todos os passos de um processo tradicional de ciência de dados. Não tem de escrever nenhum código, pode simplesmente clicar nas células abaixo para as executar e observar o resultado. Como desafio, é incentivado a experimentar este código com dados diferentes.

## Objetivo

Nesta lição, temos discutido diferentes conceitos relacionados com Ciência de Dados. Vamos tentar descobrir mais conceitos relacionados fazendo alguma **mineração de texto**. Vamos começar com um texto sobre Ciência de Dados, extrair palavras-chave deste, e depois tentar visualizar o resultado.

Como texto, vou usar a página sobre Ciência de Dados da Wikipédia:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## Passo 1: Obter os Dados

O primeiro passo em qualquer processo de ciência de dados é obter os dados. Vamos usar a biblioteca `requests` para isso:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## Passo 2: Transformar os Dados

O passo seguinte é converter os dados para a forma adequada ao processamento. No nosso caso, descarregámos código fonte HTML da página, e precisamos de o converter em texto simples.

Existem muitas maneiras de fazer isto. Vamos usar [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), uma biblioteca Python popular para parsing de HTML. O BeautifulSoup permite-nos selecionar elementos HTML específicos, para que possamos focar no conteúdo principal do artigo da Wikipédia e reduzir alguns menus de navegação, barras laterais, rodapés e outro conteúdo irrelevante (embora algum texto padrão possa ainda permanecer).


Primeiro, precisamos de instalar a biblioteca BeautifulSoup para a análise de HTML:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## Passo 3: Obter Perceções

O passo mais importante é transformar os nossos dados numa forma a partir da qual possamos tirar perceções. No nosso caso, queremos extrair palavras-chave do texto e ver quais são as mais significativas.

Vamos usar a biblioteca Python chamada [RAKE](https://github.com/aneesha/RAKE) para a extração de palavras-chave. Primeiro, vamos instalar esta biblioteca caso não esteja presente: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

A funcionalidade principal está disponível a partir do objeto `Rake`, que podemos personalizar utilizando alguns parâmetros. No nosso caso, vamos definir o comprimento mínimo de uma palavra-chave para 5 caracteres, a frequência mínima de uma palavra-chave no documento para 3, e o número máximo de palavras numa palavra-chave - para 2. Sinta-se à vontade para experimentar outros valores e observar o resultado.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


Obtivemos uma lista de termos juntamente com o grau de importância associado. Como pode ver, as disciplinas mais relevantes, como machine learning e big data, estão presentes na lista nas posições superiores.

## Passo 4: Visualizar o Resultado

As pessoas conseguem interpretar melhor os dados em forma visual. Assim, muitas vezes faz sentido visualizar os dados para retirar algumas conclusões. Podemos usar a biblioteca `matplotlib` em Python para traçar uma distribuição simples das palavras-chave com a sua relevância:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

Existe, no entanto, uma forma ainda melhor de visualizar as frequências das palavras - usando **Word Cloud** (Nuvem de Palavras). Vamos precisar de instalar outra biblioteca para desenhar a nuvem de palavras a partir da nossa lista de palavras-chave.


In [ ]:
!{sys.executable} -m pip install wordcloud

O objeto `WordCloud` é responsável por receber texto original ou uma lista pré-calculada de palavras com as suas frequências, e devolver uma imagem, que pode depois ser exibida usando `matplotlib`:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

Também podemos passar o texto original para `WordCloud` - vamos ver se conseguimos obter um resultado semelhante:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Pode ver que a nuvem de palavras agora parece mais impressionante, mas também contém muito ruído (por exemplo, palavras não relacionadas como `Retrieved on`). Além disso, obtemos menos palavras-chave que consistem em duas palavras, como *data scientist* ou *computer science*. Isto acontece porque o algoritmo RAKE faz um trabalho muito melhor ao selecionar boas palavras-chave a partir do texto. Este exemplo ilustra a importância do pré-processamento e limpeza dos dados, porque uma imagem clara no final nos permitirá tomar melhores decisões.

Neste exercício, percorremos um processo simples de extração de algum significado do texto da Wikipédia, na forma de palavras-chave e nuvem de palavras. Este exemplo é bastante simples, mas demonstra bem todos os passos típicos que um data scientist segue ao trabalhar com dados, começando pela aquisição de dados até à visualização.

No nosso curso, discutiremos todos esses passos em detalhe.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original na sua língua nativa deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas resultantes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
